# 🧹 Data Cleaning in Python — Complete Guide
### Regex · Duplicate Handling · Outlier Detection · Standardisation

---

**What you'll learn:**
- Clean messy phone numbers, emails, names, and dates using **Regex**
- Detect and resolve **exact and fuzzy duplicates**
- Identify outliers using **Z-Score, IQR, Isolation Forest, and Winsorisation**
- **Standardise** categories, numeric features, currencies, and missing values
- Build a **production-ready pipeline** using an OOP `DataCleaner` class

**Dataset used throughout:** Synthetic E-Commerce customer/order dataset (10 rows, 8 columns) with deliberately messy data.

## ⚙️ Setup: Install & Import Libraries

In [5]:
# Install required libraries (run once)
!pip install pandas numpy scipy scikit-learn fuzzywuzzy python-Levenshtein python-dateutil

  Using cached pandas-3.0.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
Using cached pandas-3.0.3-cp311-cp311-win_amd64.whl (9.9 MB)
   ---------------------------------------- 0.0/94.6 kB ? eta -:--:--
   ----------------- ---------------------- 41.0/94.6 kB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 94.6/94.6 kB 1.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ----- ---------------------------------- 0.2/1.5 MB 4.6 MB/s eta 0:00:01
   --------------- ------------------------ 0.6/1.5 MB 6.1 MB/s eta 0:00:01
   -------------------------------------- - 1.5/1.5 MB 10.6 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 9.8 MB/s eta 0:00:00


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\prach\\anaconda3\\Lib\\site-packages\\pandas\\_libs\\algos.cp311-win_amd64.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np
import re
import itertools
from datetime import datetime
from dateutil import parser as dateparser
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.ensemble import IsolationForest
from fuzzywuzzy import fuzz, process

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
print("✅ All libraries imported successfully")

✅ All libraries imported successfully


C:\Users\prach\anaconda3\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


## 📦 Load the Synthetic Dataset

This dataset contains **10 rows** with intentional data quality issues across all fields — mixed phone formats, invalid emails, duplicate rows, wrong date formats, negative amounts, and inconsistent categories.

In [34]:
data = {
    "customer_id": [101, 102, 102, 103, 104, 105, 106, 107, 108, 109],
    "name": [
        "Anand Kumar", "PRIYA SHARMA", "Priya Sharma", "  Ravi  Gupta  ",
        "Sunita-Patel", "John O'Brien", "meena iyer", "  ", "Zhang Wei", "carlos mendes"
    ],
    "phone": [
        "9876543210", "+91-98765-43211", "98765 43211", "0091 9876543212",
        "(987) 654-3213", "987-654-3214", "9999999999", "123", "+1-800-555-0100", "98765432"
    ],
    "email": [
        "anand@gmail.com", "priya@yahoo.com", "priya@yahoo.com", "ravi.gupta@",
        "sunita@domain.co.in", "john.obrien@corp.com", "MEENA@GMAIL.COM",
        "", "zhangwei@outlook.com", "carlos@@mail.com"
    ],
    "city": [
        "Mumbai", "mumbai", "MUMBAI", "New Delhi", "Ahmedabad",
        "Bengaluru", "Bangalore", "Chennai", "Hyderabad", "Pune"
    ],
    "order_amount": [1500.00, 2500.75, 2500.75, 980.00, 150000.00,
                     3200.50, 4100.25, -500.00, 1750.00, 2100.00],
    "order_date": [
        "2024-01-15", "15/01/2024", "Jan 15 2024", "2024-01-16",
        "2024-01-17", "17-01-2024", "2024-01-18", "2024-01-19",
        "2024-01-20", "20 Jan 2024"
    ],
    "product_category": [
        "Electronics", "electronics", "Electronics ", "Clothing",
        "Furniture", "furniture", "ELECTRONICS", "Clothing",
        "Electronics", "Clothing"
    ],
}

df = pd.DataFrame(data)
print(f"Shape: {df.shape}")
df

Shape: (10, 8)


,customer_id,name,phone,email,city,order_amount,order_date,product_category
0,101,Anand Kumar,9876543210,anand@gmail.com,Mumbai,1500.00,2024-01-15,Electronics
1,102,PRIYA SHARMA,+91-98765-43211,priya@yahoo.com,mumbai,2500.75,15/01/2024,electronics
2,102,Priya Sharma,98765 43211,priya@yahoo.com,MUMBAI,2500.75,Jan 15 2024,Electronics
3,103,Ravi Gupta,0091 9876543212,ravi.gupta@,New Delhi,980.00,2024-01-16,Clothing
4,104,Sunita-Patel,(987) 654-3213,sunita@domain.co.in,Ahmedabad,150000.00,2024-01-17,Furniture
5,105,John O'Brien,987-654-3214,john.obrien@corp.com,Bengaluru,3200.50,17-01-2024,furniture
6,106,meena iyer,9999999999,MEENA@GMAIL.COM,Bangalore,4100.25,2024-01-18,ELECTRONICS
7,107,,123,,Chennai,-500.00,2024-01-19,Clothing
8,108,Zhang Wei,+1-800-555-0100,zhangwei@outlook.com,Hyderabad,1750.00,2024-01-20,Electronics
9,109,carlos mendes,98765432,carlos@@mail.com,Pune,2100.00,20 Jan 2024,Clothing


In [8]:
# Quick data profile
print("=== Data Types ===")
print(df.dtypes)
print("\n=== Null Count ===")
print(df.isna().sum())
print("\n=== Basic Stats ===")
df.describe()

=== Data Types ===
customer_id           int64
name                 object
phone                object
email                object
city                 object
order_amount        float64
order_date           object
product_category     object
dtype: object

=== Null Count ===
customer_id         0
name                0
phone               0
email               0
city                0
order_amount        0
order_date          0
product_category    0
dtype: int64

=== Basic Stats ===


,customer_id,order_amount
count,10.000000,10.000000
mean,104.700000,16813.225000
std,2.750757,46813.637973
min,101.000000,-500.000000
25%,102.250000,1562.500000
50%,104.500000,2300.375000
75%,106.750000,3025.562500
max,109.000000,150000.000000


---
# 🔤 Chapter 1: Regex (Regular Expressions)

Regex is the **Swiss Army knife** for cleaning unstructured text fields. It lets you extract, validate, and transform data using pattern matching.

### Quick Regex Reference
| Pattern | Meaning | Example |
|---------|---------|--------|
| `\d` | Any digit (0–9) | `9` in `9876` |
| `\D` | Non-digit | `a` in `a123` |
| `\w` | Word char | `A`, `9`, `_` |
| `\s` | Whitespace | space, tab |
| `+` | One or more | `\d+` matches `123` |
| `?` | Zero or one | `colou?r` |
| `{m,n}` | Between m–n times | `\d{10}` |
| `^` | Start of string | `^\d` starts with digit |
| `$` | End of string | `\d$` ends with digit |
| `[...]` | Character class | `[aeiou]` any vowel |

### 1.1 Use Case 1 — Phone Number Cleaning & Validation
**Industry: Telecom / Banking CRM**

Goal: strip all non-digit characters, remove country code prefixes, validate as 10-digit Indian number, return in E.164 format (`+91XXXXXXXXXX`).

In [31]:
def clean_phone(raw, default_cc="91"):
    """
    Standardise Indian phone numbers to E.164 format.
    Returns +91XXXXXXXXXX or None if invalid.
    """
    if pd.isna(raw) or str(raw).strip() == "":
        return None

    # Step 1: Remove all non-digit characters
    digits = re.sub(r'\D', '', str(raw))

    # Step 2: Strip country code prefixes
    if digits.startswith("91") and len(digits) == 12:
        digits = digits[2:]              # +91 / 91
    elif digits.startswith("0091") and len(digits) == 14:
        digits = digits[4:]              # 0091
    elif digits.startswith("1") and len(digits) == 11:
        return f"+{digits}"              # US number - keep as-is

    # Step 3: Validate — 10 digits starting with 6/7/8/9
    if re.match(r'^[6-9]\d{9}$', digits):
        return f"+{default_cc}{digits}"
    return None


df["phone_cleaned"] = df["phone"].apply(clean_phone)
df["phone_valid"]   = df["phone_cleaned"].notna()

df[["phone", "phone_cleaned", "phone_valid"]]

,phone,phone_cleaned,phone_valid
0,9876543210,+919876543210,True
1,+91-98765-43211,+919876543211,True
2,98765 43211,+919876543211,True
3,0091 9876543212,+919876543212,True
4,(987) 654-3213,+919876543213,True
5,987-654-3214,+919876543214,True
6,9999999999,+919999999999,True
7,123,None,False
8,+1-800-555-0100,+18005550100,True
9,98765432,None,False


### 1.2 Use Case 2 — Email Validation & Domain Extraction
**Industry: Marketing / SaaS**

In [10]:
EMAIL_REGEX = re.compile(
    r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'
)

PERSONAL_DOMAINS = {"gmail.com", "yahoo.com", "hotmail.com", "outlook.com"}

def validate_email(email):
    """Normalise, validate, extract domain, classify B2C/B2B."""
    if pd.isna(email) or str(email).strip() == "":
        return None, False, None, "Unknown"

    email = str(email).strip().lower()

    if email.count('@') != 1:
        return email, False, None, "Invalid"

    is_valid = bool(EMAIL_REGEX.match(email))
    domain   = email.split('@')[1] if is_valid else None
    email_type = "B2C" if domain in PERSONAL_DOMAINS else ("B2B" if domain else "Invalid")

    return email, is_valid, domain, email_type


df[["email_clean", "email_valid", "email_domain", "email_type"]] = df["email"].apply(
    lambda e: pd.Series(validate_email(e))
)

df[["email", "email_clean", "email_valid", "email_type"]]

,email,email_clean,email_valid,email_type
0,anand@gmail.com,anand@gmail.com,True,B2C
1,priya@yahoo.com,priya@yahoo.com,True,B2C
2,priya@yahoo.com,priya@yahoo.com,True,B2C
3,ravi.gupta@,ravi.gupta@,False,Invalid
4,sunita@domain.co.in,sunita@domain.co.in,True,B2B
5,john.obrien@corp.com,john.obrien@corp.com,True,B2B
6,MEENA@GMAIL.COM,meena@gmail.com,True,B2C
7,,None,False,Unknown
8,zhangwei@outlook.com,zhangwei@outlook.com,True,B2C
9,carlos@@mail.com,carlos@@mail.com,False,Invalid


### 1.3 Use Case 3 — Name Cleaning & Parsing
**Industry: Banking KYC / HR Systems**

In [36]:
def clean_name(raw):
    """Remove extra whitespace, title-case, split into first/last."""
    if pd.isna(raw) or str(raw).strip() == "":
        return None, None, None

    name = str(raw).strip()
    name = re.sub(r"[-']"," ",name)
    name = re.sub(r'\s+', ' ', name)                          # collapse spaces
    name = re.sub(r"[A-Za-z]+(['-][A-Za-z]+)*",              # title-case incl. hyphenated
                  lambda m: m.group().title(), name)

    parts = name.split()
    first = parts[0] if len(parts) >= 1 else None
    last  = parts[-1] if len(parts) >= 2 else None
    return name, first, last


df[["name_clean", "first_name", "last_name"]] = df["name"].apply(
    lambda n: pd.Series(clean_name(n))
)

df[["name", "name_clean", "first_name", "last_name"]]

,name,name_clean,first_name,last_name
0,Anand Kumar,Anand Kumar,Anand,Kumar
1,PRIYA SHARMA,Priya Sharma,Priya,Sharma
2,Priya Sharma,Priya Sharma,Priya,Sharma
3,Ravi Gupta,Ravi Gupta,Ravi,Gupta
4,Sunita-Patel,Sunita Patel,Sunita,Patel
5,John O'Brien,John O Brien,John,Brien
6,meena iyer,Meena Iyer,Meena,Iyer
7,,None,None,None
8,Zhang Wei,Zhang Wei,Zhang,Wei
9,carlos mendes,Carlos Mendes,Carlos,Mendes


### 1.4 Use Case 4 — Date Parsing from Mixed Formats
**Industry: Supply Chain / Finance**

In [12]:
DATE_PATTERNS = [
    (r'^\d{4}-\d{2}-\d{2}$',  '%Y-%m-%d'),   # 2024-01-15
    (r'^\d{2}/\d{2}/\d{4}$',  '%d/%m/%Y'),   # 15/01/2024
    (r'^\d{2}-\d{2}-\d{4}$',  '%d-%m-%Y'),   # 17-01-2024
]

def parse_date(raw):
    """Parse dates from any format, return (date, method)."""
    if pd.isna(raw):
        return None, "missing"

    raw = str(raw).strip()

    # Try deterministic pattern matching first
    for pattern, fmt in DATE_PATTERNS:
        if re.match(pattern, raw):
            try:
                return datetime.strptime(raw, fmt).date(), "regex_matched"
            except ValueError:
                pass

    # Fallback to dateutil fuzzy parser
    try:
        return dateparser.parse(raw, dayfirst=True).date(), "fuzzy_parsed"
    except Exception:
        return None, "parse_error"


df[["order_date_clean", "date_parse_method"]] = df["order_date"].apply(
    lambda d: pd.Series(parse_date(d))
)

df[["order_date", "order_date_clean", "date_parse_method"]]

,order_date,order_date_clean,date_parse_method
0,2024-01-15,2024-01-15,regex_matched
1,15/01/2024,2024-01-15,regex_matched
2,Jan 15 2024,2024-01-15,fuzzy_parsed
3,2024-01-16,2024-01-16,regex_matched
4,2024-01-17,2024-01-17,regex_matched
5,17-01-2024,2024-01-17,regex_matched
6,2024-01-18,2024-01-18,regex_matched
7,2024-01-19,2024-01-19,regex_matched
8,2024-01-20,2024-01-20,regex_matched
9,20 Jan 2024,2024-01-20,fuzzy_parsed


### 1.5 Use Case 5 — Extract Amounts from Free Text
**Industry: Insurance Claims / Procurement**

In [13]:
claim_notes = [
    "Claim approved for Rs. 45,000 after deductible",
    "Reimbursement of INR 1,23,456.78 processed",
    "Amount: $2,500 USD covered",
    "No claim amount mentioned",
    "Paid ₹78000 to hospital",
]

def extract_amount(text):
    """Extract first numeric amount regardless of currency symbol."""
    if not text:
        return None
    pattern = r'(?:Rs\.?|INR|\$|₹)?\s*([\d,]+(?:\.\d{1,2})?)'
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        return float(match.group(1).replace(',', ''))
    return None


results = pd.DataFrame({
    "note": claim_notes,
    "extracted_amount": [extract_amount(n) for n in claim_notes]
})
results

,note,extracted_amount
0,"Claim approved for Rs. 45,000 after deductible",45000.00
1,"Reimbursement of INR 1,23,456.78 processed",123456.78
2,"Amount: $2,500 USD covered",2500.00
3,No claim amount mentioned,NaN
4,Paid ₹78000 to hospital,78000.00


---
# 👥 Chapter 2: Duplicate Detection & Handling

Duplicates are **silent killers**. A customer counted twice inflates churn; a transaction recorded twice doubles revenue. Two types:
- **Exact duplicates** — identical rows
- **Fuzzy duplicates** — same entity, slightly different representation

### 2.1 Exact Duplicate Detection
**Industry: Retail / Banking — double-click or network retry creating duplicate orders**

In [14]:
# Fully identical rows
exact_dupes = df[df.duplicated(keep=False)]
print(f"Fully identical rows: {len(exact_dupes)}")

# Key-level duplicates (same customer + amount = likely double-submit)
key_dupes = df[df.duplicated(subset=["customer_id", "order_amount"], keep=False)]
print(f"Key-level duplicates: {len(key_dupes)}")
print(key_dupes[["customer_id", "name", "order_amount"]])

Fully identical rows: 0
Key-level duplicates: 2
   customer_id          name  order_amount
1          102  PRIYA SHARMA       2500.75
2          102  Priya Sharma       2500.75


In [15]:
# Tag before removing (audit trail — never silently delete!)
df["is_duplicate"] = df.duplicated(subset=["customer_id", "order_amount"], keep="first")

# Remove duplicates
df_deduped = df[~df["is_duplicate"]].reset_index(drop=True)
print(f"Before dedup: {len(df)} rows | After: {len(df_deduped)} rows")

df[["customer_id", "name", "order_amount", "is_duplicate"]]

Before dedup: 10 rows | After: 9 rows


,customer_id,name,order_amount,is_duplicate
0,101,Anand Kumar,1500.00,False
1,102,PRIYA SHARMA,2500.75,False
2,102,Priya Sharma,2500.75,True
3,103,Ravi Gupta,980.00,False
4,104,Sunita-Patel,150000.00,False
5,105,John O'Brien,3200.50,False
6,106,meena iyer,4100.25,False
7,107,,-500.00,False
8,108,Zhang Wei,1750.00,False
9,109,carlos mendes,2100.00,False


### 2.2 Fuzzy Duplicate Detection
**Industry: Hospitality / Healthcare — 'Bangalore' vs 'Bengaluru', 'Ravi Gupta' vs 'R. Gupta'**

In [16]:
def find_fuzzy_dupes(name_list, threshold=80):
    """
    Compare all name pairs, flag those above similarity threshold.
    Uses token_sort_ratio to handle word-order differences.
    """
    pairs = []
    for a, b in itertools.combinations(name_list, 2):
        score = fuzz.token_sort_ratio(a, b)
        if score >= threshold:
            pairs.append({"name_a": a, "name_b": b, "similarity": score})
    return pd.DataFrame(pairs)


names = df["name_clean"].dropna().tolist()
fuzzy_result = find_fuzzy_dupes(names, threshold=80)
print(f"Fuzzy duplicate pairs found: {len(fuzzy_result)}")
fuzzy_result

Fuzzy duplicate pairs found: 1


,name_a,name_b,similarity
0,Priya Sharma,Priya Sharma,100


### 2.3 City Canonicalisation via Fuzzy Matching

In [17]:
CANONICAL_CITIES = [
    "Mumbai", "New Delhi", "Ahmedabad", "Bengaluru",
    "Chennai", "Hyderabad", "Pune", "Kolkata"
]

def standardise_city(raw):
    if pd.isna(raw) or str(raw).strip() == "":
        return None, 0
    raw = str(raw).strip()
    # Exact case-insensitive match first
    for city in CANONICAL_CITIES:
        if raw.lower() == city.lower():
            return city, 100
    # Fuzzy fallback
    best, score = process.extractOne(raw, CANONICAL_CITIES, scorer=fuzz.token_sort_ratio)
    return (best, score) if score >= 75 else (raw.title(), score)


df[["city_std", "city_match_score"]] = df["city"].apply(
    lambda c: pd.Series(standardise_city(c))
)

df[["city", "city_std", "city_match_score"]]
# Notice: 'Bangalore' correctly maps to 'Bengaluru' (official name)

,city,city_std,city_match_score
0,Mumbai,Mumbai,100
1,mumbai,Mumbai,100
2,MUMBAI,Mumbai,100
3,New Delhi,New Delhi,100
4,Ahmedabad,Ahmedabad,100
5,Bengaluru,Bengaluru,100
6,Bangalore,Bangalore,67
7,Chennai,Chennai,100
8,Hyderabad,Hyderabad,100
9,Pune,Pune,100


### 2.4 Cross-Dataset Record Linkage
**Industry: Government / Insurance — match records across CRM and billing without a common key**

In [18]:
# Two systems with the same people, different representations
crm_df = pd.DataFrame({
    "name":  ["Anand Kumar", "Priya Sharma", "Ravi Gupta"],
    "phone": ["+919876543210", "+919876543211", "+919876543212"],
    "email": ["anand@gmail.com", "priya@yahoo.com", "ravi@corp.com"],
})

billing_df = pd.DataFrame({
    "full_name": ["ANAND KUMAR", "P. Sharma", "Ravi  Gupta"],
    "mobile":    ["9876543210", "98765 43211", "91-9876543212"],
    "mail":      ["anand@gmail.com", "priya@yahoo.com", "ravi@corp.com"],
})

# Normalise name keys
def norm(n): return re.sub(r'\s+', ' ', str(n)).strip().lower()
crm_df["name_key"]     = crm_df["name"].apply(norm)
billing_df["name_key"] = billing_df["full_name"].apply(norm)

# Score-based matching
def match_score(r_a, r_b):
    name_sim  = fuzz.token_sort_ratio(r_a["name_key"], r_b["name_key"]) / 100
    email_sim = 1.0 if r_a["email"] == r_b["mail"] else 0.0
    return name_sim * 0.5 + email_sim * 0.5

matches = []
for _, r_crm in crm_df.iterrows():
    for _, r_bill in billing_df.iterrows():
        score = match_score(r_crm, r_bill)
        if score >= 0.7:
            matches.append({
                "crm_name": r_crm["name"],
                "billing_name": r_bill["full_name"],
                "match_score": round(score, 2)
            })

pd.DataFrame(matches)

,crm_name,billing_name,match_score
0,Anand Kumar,ANAND KUMAR,1.0
1,Priya Sharma,P. Sharma,0.9
2,Ravi Gupta,Ravi Gupta,1.0


---
# 📊 Chapter 3: Outlier Detection

> **Always ask WHY before removing an outlier.** In fraud detection, the outlier IS the signal. In sensor data, it may be a faulty reading.

| Type | Example | Action |
|------|---------|--------|
| Data entry error | Negative age | Remove or correct |
| Genuine extreme | VIP ₹10L order | Keep, flag, segment |
| Fraud signal | 3AM transaction | Escalate |
| Ambiguous | Unusual but possible | Cap (Winsorise) |

### 3.1 Z-Score Method (Parametric — assumes Normal distribution)
**Industry: Retail / Finance — anomalous transaction amounts**

In [37]:
df["amount_zscore"]  = stats.zscore(df["order_amount"])
df["outlier_zscore"] = df["amount_zscore"].abs() > 2.5   # strict threshold

print("Outliers detected by Z-Score:")
df[df["outlier_zscore"]][["customer_id", "name_clean", "order_amount", "amount_zscore"]]

Outliers detected by Z-Score:


,customer_id,name_clean,order_amount,amount_zscore
4,104,Sunita Patel,150000.0,2.998938


### 3.2 IQR Method (Non-parametric — robust to skewed data)
**Industry: E-Commerce — right-skewed order amounts**

In [38]:
def detect_outliers_iqr(series, multiplier=1.5):
    """
    IQR-based outlier detection.
    multiplier=1.5 → mild outliers (standard)
    multiplier=3.0 → extreme outliers only
    """
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    return (series < lower) | (series > upper), lower, upper


df["outlier_iqr"], lower, upper = detect_outliers_iqr(df["order_amount"])
print(f"IQR fences → Lower: ₹{lower:.2f}  |  Upper: ₹{upper:.2f}")

# Business rule outliers (domain knowledge)
df["outlier_business"] = (df["order_amount"] <= 0) | (df["order_amount"] > 100000)

print("\nIQR Outliers:")
print(df[df["outlier_iqr"]][["customer_id", "name_clean", "order_amount"]])
print("\nBusiness Rule Violations:")
print(df[df["outlier_business"]][["customer_id", "name_clean", "order_amount"]])

IQR fences → Lower: ₹-632.09  |  Upper: ₹5220.16

IQR Outliers:
   customer_id    name_clean  order_amount
4          104  Sunita Patel      150000.0

Business Rule Violations:
   customer_id    name_clean  order_amount
4          104  Sunita Patel      150000.0
7          107          None        -500.0


### 3.3 Isolation Forest — ML-Based Multivariate Detection
**Industry: Fraud Detection / Cybersecurity — detect anomalies across multiple features**

In [21]:
np.random.seed(42)
n = 500

# Simulate 490 normal + 10 fraudulent transactions
transactions = pd.DataFrame({
    "amount":      np.concatenate([np.random.normal(2000, 500, 490),
                                   np.random.uniform(50000, 200000, 10)]),
    "frequency":   np.concatenate([np.random.randint(1, 10, 490),
                                   np.random.randint(50, 200, 10)]),
    "hour_of_day": np.concatenate([np.random.randint(8, 22, 490),
                                   np.random.randint(1, 4, 10)]),  # late-night anomalies
})

iso = IsolationForest(
    n_estimators=100,
    contamination=0.05,   # expect ~5% anomalies
    random_state=42
)
transactions["anomaly_flag"] = iso.fit_predict(
    transactions[["amount", "frequency", "hour_of_day"]]
)  # -1 = anomaly, 1 = normal
transactions["is_anomaly"] = transactions["anomaly_flag"] == -1

anomalies = transactions[transactions["is_anomaly"]]
print(f"Total: {len(transactions)} | Anomalies: {len(anomalies)} ({len(anomalies)/len(transactions)*100:.1f}%)")
print("\nAnomaly summary:")
anomalies[["amount", "frequency", "hour_of_day"]].describe().round(1)

C:\Users\prach\anaconda3\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


Total: 500 | Anomalies: 25 (5.0%)

Anomaly summary:


,amount,frequency,hour_of_day
count,25.0,25.0,25.0
mean,47965.6,57.6,10.2
std,61295.8,70.5,7.9
min,379.4,1.0,1.0
25%,1143.4,4.0,3.0
50%,3360.1,9.0,9.0
75%,104462.3,110.0,19.0
max,168375.7,192.0,21.0


### 3.4 Winsorisation (Percentile Capping)
**Industry: Credit Scoring / Insurance — cap extremes without losing records**

In [22]:
def winsorise(series, lower_pct=0.01, upper_pct=0.99):
    """
    Cap extreme values at percentile boundaries.
    Preferred when:
      - Outliers are REAL but extreme (not errors)
      - You cannot afford to lose records
      - Model is sensitive to scale (linear regression)
    """
    lower_bound = series.quantile(lower_pct)
    upper_bound = series.quantile(upper_pct)
    capped = series.clip(lower=lower_bound, upper=upper_bound)

    print(f"Capped below {lower_pct*100:.0f}th pct (₹{lower_bound:.2f}): "
          f"{(series < lower_bound).sum()} values")
    print(f"Capped above {upper_pct*100:.0f}th pct (₹{upper_bound:.2f}): "
          f"{(series > upper_bound).sum()} values")
    return capped


df["order_amount_winsorised"] = winsorise(df["order_amount"])
df[["order_amount", "order_amount_winsorised"]]

Capped below 1th pct (₹-366.80): 1 values
Capped above 99th pct (₹136869.02): 1 values


,order_amount,order_amount_winsorised
0,1500.00,1500.0000
1,2500.75,2500.7500
2,2500.75,2500.7500
3,980.00,980.0000
4,150000.00,136869.0225
5,3200.50,3200.5000
6,4100.25,4100.2500
7,-500.00,-366.8000
8,1750.00,1750.0000
9,2100.00,2100.0000


---
# 🔧 Chapter 4: Data Standardisation

Standardisation = bringing data into a **consistent, comparable format**:
1. **Format standardisation** — dates, phones, categories  
2. **Value normalisation** — Min-Max, Z-Score scaling for ML  
3. **Unit harmonisation** — currencies, units of measure

### 4.1 Categorical Standardisation
**Industry: Retail Analytics — 'electronics', 'Electronics ', 'ELECTRONICS' → 'Electronics'**

In [23]:
# Simple case + strip
df["category_std"] = df["product_category"].str.strip().str.title()
print("After strip + title case:")
print(df[["product_category", "category_std"]].drop_duplicates())

# Synonym mapping (domain knowledge)
CATEGORY_MAP = {
    "Electronics": "Electronics",
    "Electronic":  "Electronics",
    "Clothing":    "Apparel",
    "Clothes":     "Apparel",
    "Furniture":   "Home & Furniture",
    "Home":        "Home & Furniture",
}

def map_category(raw):
    if pd.isna(raw): return "Unknown"
    return CATEGORY_MAP.get(str(raw).strip().title(), str(raw).strip().title())

df["category_mapped"] = df["product_category"].apply(map_category)
print("\nAfter synonym mapping:")
df[["product_category", "category_mapped"]].drop_duplicates()

After strip + title case:
  product_category category_std
0      Electronics  Electronics
1      electronics  Electronics
2     Electronics   Electronics
3         Clothing     Clothing
4        Furniture    Furniture
5        furniture    Furniture
6      ELECTRONICS  Electronics

After synonym mapping:


,product_category,category_mapped
0,Electronics,Electronics
1,electronics,Electronics
2,Electronics,Electronics
3,Clothing,Apparel
4,Furniture,Home & Furniture
5,furniture,Home & Furniture
6,ELECTRONICS,Electronics


### 4.2 Numeric Scaling for Machine Learning

| Scaler | Formula | Best For |
|--------|---------|----------|
| MinMaxScaler | (x - min) / (max - min) | Neural nets, KNN |
| StandardScaler | (x - mean) / std | Linear regression, PCA |
| RobustScaler | (x - median) / IQR | Data with outliers |

In [24]:
features = pd.DataFrame({
    "age":           [25, 34, 45, 28, 52, 61, 29, 38, 47, 33],
    "annual_income": [350000, 850000, 1200000, 480000, 950000,
                      1500000, 420000, 780000, 1100000, 620000],
    "credit_score":  [650, 720, 810, 680, 750, 790, 700, 740, 800, 710],
})

scalers = {
    "minmax":  MinMaxScaler(),
    "std":     StandardScaler(),
    "robust":  RobustScaler(),
}

scaled_dfs = {}
for name, scaler in scalers.items():
    scaled_dfs[name] = pd.DataFrame(
        scaler.fit_transform(features),
        columns=features.columns
    )

# Compare age column across scalers
comparison = pd.DataFrame({
    "original":  features["age"],
    "minmax":    scaled_dfs["minmax"]["age"].round(3),
    "z-score":   scaled_dfs["std"]["age"].round(3),
    "robust":    scaled_dfs["robust"]["age"].round(3),
})
comparison

,original,minmax,z-score,robust
0,25,0.000,-1.280,-0.667
1,34,0.250,-0.469,-0.121
2,45,0.556,0.523,0.545
3,28,0.083,-1.009,-0.485
4,52,0.750,1.153,0.970
5,61,1.000,1.964,1.515
6,29,0.111,-0.919,-0.424
7,38,0.361,-0.108,0.121
8,47,0.611,0.703,0.667
9,33,0.222,-0.559,-0.182


In [25]:
# ⚠️ CRITICAL: Always fit on TRAIN set only, then transform both train and test
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(features, test_size=0.2, random_state=42)
scaler = StandardScaler()

scaler.fit(X_train)                        # ← fit ONLY on training data
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # ← use SAME fit for test

print("Train scaled shape:", X_train_scaled.shape)
print("Test scaled shape: ", X_test_scaled.shape)
print("\n✅ No data leakage — scaler fitted only on training data")

Train scaled shape: (8, 3)
Test scaled shape:  (2, 3)

✅ No data leakage — scaler fitted only on training data


### 4.3 Currency Standardisation
**Industry: International E-Commerce — USD, EUR, INR from different regional systems**

In [26]:
# Static FX rates (production: use live API e.g. exchangeratesapi.io)
FX_RATES_TO_INR = {
    "INR": 1.0,
    "USD": 83.5,
    "EUR": 90.2,
    "GBP": 105.0,
    "AED": 22.7,
}

multi_curr = pd.DataFrame({
    "sale_id":  [1, 2, 3, 4, 5],
    "amount":   [1500, 250, 200, 1000, 5000],
    "currency": ["INR", "USD", "EUR", "AED", "INR"],
})

def convert_to_inr(row):
    rate = FX_RATES_TO_INR.get(row["currency"])
    if rate is None:
        return None, f"Unknown currency: {row['currency']}"
    return round(row["amount"] * rate, 2), "OK"

multi_curr[["amount_inr", "fx_status"]] = multi_curr.apply(
    lambda r: pd.Series(convert_to_inr(r)), axis=1
)
multi_curr

,sale_id,amount,currency,amount_inr,fx_status
0,1,1500,INR,1500.0,OK
1,2,250,USD,20875.0,OK
2,3,200,EUR,18040.0,OK
3,4,1000,AED,22700.0,OK
4,5,5000,INR,5000.0,OK


### 4.4 Missing Value Standardisation
**Industry: Healthcare / Surveys — -999, 'N/A', 'none', '', NULL all mean missing**

In [27]:
MISSING_MARKERS = [-999, -1, "N/A", "NA", "n/a", "none", "None", "NONE",
                   "null", "NULL", "nan", "NaN", "unknown", "", " "]

def standardise_nulls(df):
    """Replace all common missing markers with np.nan."""
    return df.replace(MISSING_MARKERS, np.nan)

def smart_fill(df):
    """Impute missing values based on data type and missingness rate."""
    df = df.copy()
    for col in df.columns:
        null_pct = df[col].isna().mean() * 100
        if null_pct == 0:
            continue
        elif null_pct > 60:
            print(f"  ⚠️  DROP '{col}' — {null_pct:.1f}% missing")
            df.drop(columns=[col], inplace=True)
        elif df[col].dtype in ["float64", "int64"]:
            med = df[col].median()
            df[col].fillna(med, inplace=True)
            print(f"  ✅ FILL '{col}' with median {med:.2f} ({null_pct:.1f}% missing)")
        else:
            mode_val = df[col].mode()[0] if not df[col].mode().empty else "Unknown"
            df[col].fillna(mode_val, inplace=True)
            print(f"  ✅ FILL '{col}' with mode '{mode_val}' ({null_pct:.1f}% missing)")
    return df

# Demo
demo = pd.DataFrame({
    "age":    [25, -999, 30, np.nan, 45],
    "income": [50000, 70000, "N/A", 60000, None],
    "city":   ["Mumbai", "", "Delhi", "NA", "Pune"],
})

print("Before:")
print(demo)
demo_clean = standardise_nulls(demo)
demo_clean["age"] = pd.to_numeric(demo_clean["age"], errors="coerce")
demo_clean["income"] = pd.to_numeric(demo_clean["income"], errors="coerce")
print("\nAfter filling:")
smart_fill(demo_clean)

Before:
     age income    city
0   25.0  50000  Mumbai
1 -999.0  70000        
2   30.0    N/A   Delhi
3    NaN  60000      NA
4   45.0   None    Pune

After filling:
  ✅ FILL 'age' with median 30.00 (40.0% missing)
  ✅ FILL 'income' with median 60000.00 (40.0% missing)
  ✅ FILL 'city' with mode 'Delhi' (40.0% missing)


C:\Users\prach\AppData\Local\Temp\ipykernel_14812\1642058781.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return df.replace(MISSING_MARKERS, np.nan)
C:\Users\prach\AppData\Local\Temp\ipykernel_14812\1642058781.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(med, inplace=Tr

,age,income,city
0,25.0,50000.0,Mumbai
1,30.0,70000.0,Delhi
2,30.0,60000.0,Delhi
3,30.0,60000.0,Delhi
4,45.0,60000.0,Pune


---
# 🏭 Chapter 5: End-to-End Production Pipeline

Combining all four chapters into a single, **production-ready `DataCleaner` class** with:
- Method chaining (fluent API)
- Audit log for every transformation
- Before/after counts

In [28]:
class DataCleaner:
    """
    Production-grade data cleaning pipeline for e-commerce customer/order data.
    Supports method chaining: DataCleaner(df).clean_names().clean_phones().run()
    """

    CANONICAL_CITIES = ["Mumbai", "New Delhi", "Ahmedabad", "Bengaluru",
                        "Chennai", "Hyderabad", "Pune", "Kolkata"]
    MISSING_MARKERS  = [-999, -1, "N/A", "NA", "none", "NULL", "", " "]
    EMAIL_REGEX      = re.compile(r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$')

    def __init__(self, df):
        self.df  = df.copy()
        self.log = []

    def _log(self, step, detail):
        self.log.append({"step": step, "detail": detail})
        print(f"  [✓] {step:<20} {detail}")

    # ── Step 1: Null standardisation ──────────────────────────────────────
    def standardise_nulls(self):
        before = self.df.isna().sum().sum()
        self.df.replace(self.MISSING_MARKERS, np.nan, inplace=True)
        after = self.df.isna().sum().sum()
        self._log("NULL_STD", f"NaN count: {before} → {after}")
        return self

    # ── Step 2: Name cleaning ─────────────────────────────────────────────
    def clean_names(self):
        def _clean(raw):
            if pd.isna(raw): return None
            n = re.sub(r'\s+', ' ', str(raw).strip())
            return re.sub(r"[A-Za-z]+(['-][A-Za-z]+)*",
                          lambda m: m.group().title(), n)
        self.df["name_clean"] = self.df["name"].apply(_clean)
        self._log("NAME_CLEAN", f"{self.df['name_clean'].notna().sum()} names cleaned")
        return self

    # ── Step 3: Phone cleaning ────────────────────────────────────────────
    def clean_phones(self):
        def _clean(raw):
            if pd.isna(raw): return None
            d = re.sub(r'\D', '', str(raw))
            if d.startswith("91") and len(d) == 12:   d = d[2:]
            elif d.startswith("0091") and len(d) == 14: d = d[4:]
            return f"+91{d}" if re.match(r'^[6-9]\d{9}$', d) else None
        self.df["phone_clean"] = self.df["phone"].apply(_clean)
        v = self.df["phone_clean"].notna().sum()
        self._log("PHONE_CLEAN", f"{v}/{len(self.df)} phones valid")
        return self

    # ── Step 4: Email cleaning ────────────────────────────────────────────
    def clean_emails(self):
        def _clean(raw):
            if pd.isna(raw) or str(raw).count('@') != 1: return None
            e = str(raw).strip().lower()
            return e if self.EMAIL_REGEX.match(e) else None
        self.df["email_clean"] = self.df["email"].apply(_clean)
        v = self.df["email_clean"].notna().sum()
        self._log("EMAIL_CLEAN", f"{v}/{len(self.df)} emails valid")
        return self

    # ── Step 5: City standardisation ──────────────────────────────────────
    def standardise_cities(self):
        def _std(raw):
            if pd.isna(raw): return None
            match, score = process.extractOne(
                str(raw).strip(), self.CANONICAL_CITIES, scorer=fuzz.token_sort_ratio)
            return match if score >= 75 else str(raw).strip().title()
        self.df["city_std"] = self.df["city"].apply(_std)
        self._log("CITY_STD", "Cities mapped to canonical list")
        return self

    # ── Step 6: Deduplication ─────────────────────────────────────────────
    def remove_duplicates(self, subset=None):
        before = len(self.df)
        self.df["is_duplicate"] = self.df.duplicated(subset=subset, keep="first")
        self.df = self.df[~self.df["is_duplicate"]].reset_index(drop=True)
        after = len(self.df)
        self._log("DEDUP", f"Removed {before - after} dupes | {after} rows remain")
        return self

    # ── Step 7: Outlier handling ──────────────────────────────────────────
    def handle_outliers(self, col, method="iqr", action="flag"):
        series = self.df[col]
        if method == "iqr":
            Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
            mask = (series < Q1 - 1.5*(Q3-Q1)) | (series > Q3 + 1.5*(Q3-Q1))
        elif method == "zscore":
            mask = stats.zscore(series).abs() > 2.5
        elif method == "business":
            mask = (series <= 0) | (series > 100000)
        else:
            mask = pd.Series([False]*len(self.df))

        self.df[f"{col}_outlier"] = mask
        if action == "cap":
            lo, hi = series.quantile(0.01), series.quantile(0.99)
            self.df[col] = series.clip(lo, hi)
            self._log("OUTLIER_CAP", f"{col}: {mask.sum()} capped")
        else:
            self._log("OUTLIER_FLAG", f"{col}: {mask.sum()} flagged")
        return self

    # ── Step 8: Category standardisation ─────────────────────────────────
    def standardise_categories(self, col, mapping=None):
        self.df[f"{col}_std"] = self.df[col].str.strip().str.title()
        if mapping:
            self.df[f"{col}_std"] = self.df[f"{col}_std"].replace(mapping)
        self._log("CAT_STD", f"{col} standardised")
        return self

    # ── Full pipeline ─────────────────────────────────────────────────────
    def run(self):
        return (self
            .standardise_nulls()
            .clean_names()
            .clean_phones()
            .clean_emails()
            .standardise_cities()
            .remove_duplicates(subset=["customer_id", "order_amount"])
            .handle_outliers("order_amount", method="business", action="flag")
            .standardise_categories("product_category")
        )

    def summary(self):
        print("\n" + "="*60)
        print("  CLEANING PIPELINE SUMMARY")
        print("="*60)
        for entry in self.log:
            print(f"  {entry['step']:<22} {entry['detail']}")
        print(f"\n  Final shape : {self.df.shape}")
        print(f"  Null %      : {self.df.isna().mean().mean()*100:.1f}%")
        print("="*60)
        return self.df


print("✅ DataCleaner class defined")

✅ DataCleaner class defined


In [29]:
# ── Run the full pipeline ─────────────────────────────────────────────────
raw_df = pd.DataFrame(data)   # fresh copy
cleaner = DataCleaner(raw_df)
df_final = cleaner.run().summary()

  [✓] NULL_STD             NaN count: 0 → 1
  [✓] NAME_CLEAN           10 names cleaned
  [✓] PHONE_CLEAN          7/10 phones valid
  [✓] EMAIL_CLEAN          7/10 emails valid
  [✓] CITY_STD             Cities mapped to canonical list
  [✓] DEDUP                Removed 1 dupes | 9 rows remain
  [✓] OUTLIER_FLAG         order_amount: 2 flagged
  [✓] CAT_STD              product_category standardised

  CLEANING PIPELINE SUMMARY
  NULL_STD               NaN count: 0 → 1
  NAME_CLEAN             10 names cleaned
  PHONE_CLEAN            7/10 phones valid
  EMAIL_CLEAN            7/10 emails valid
  CITY_STD               Cities mapped to canonical list
  DEDUP                  Removed 1 dupes | 9 rows remain
  OUTLIER_FLAG           order_amount: 2 flagged
  CAT_STD                product_category standardised

  Final shape : (9, 15)
  Null %      : 5.2%


In [ ]:
# View the clean output
df_final[[
    "customer_id", "name_clean", "phone_clean",
    "email_clean", "city_std", "order_amount",
    "order_amount_outlier", "product_category_std"
]]

---
# 📋 Chapter 6: Quick Reference Cheat Sheet

### 6.1 Regex Patterns for Common Indian Data Fields

| Field | Pattern | Notes |
|-------|---------|-------|
| Indian Phone | `^[6-9]\d{9}$` | 10-digit, starts 6/7/8/9 |
| Email | `^[\w.%+\-]+@[\w.\-]+\.[a-zA-Z]{2,}$` | Most common formats |
| PAN Card | `^[A-Z]{5}[0-9]{4}[A-Z]{1}$` | 10-char PAN |
| Aadhar | `^[2-9]{1}[0-9]{11}$` | 12 digits |
| Pincode | `^[1-9][0-9]{5}$` | 6-digit Indian pincode |
| GST Number | `^[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z]{1}[1-9A-Z]{1}Z[0-9A-Z]{1}$` | 15-char GST |

In [ ]:
# Test all patterns
PATTERNS = {
    "Indian Phone":  r'^[6-9]\d{9}$',
    "PAN Card":      r'^[A-Z]{5}[0-9]{4}[A-Z]{1}$',
    "Aadhar":        r'^[2-9]{1}[0-9]{11}$',
    "Pincode":       r'^[1-9][0-9]{5}$',
    "GST Number":    r'^[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z]{1}[1-9A-Z]{1}Z[0-9A-Z]{1}$',
}

test_values = {
    "Indian Phone":  ["9876543210", "1234567890", "98765432"],
    "PAN Card":      ["ABCDE1234F", "ABCDE1234", "abcde1234f"],
    "Aadhar":        ["234567891234", "1234567890123", "23456789123"],
    "Pincode":       ["400001", "001234", "4000011"],
    "GST Number":    ["27ABCDE1234F1Z5", "27ABCDE1234F", "27abcde1234f1z5"],
}

for field, pattern in PATTERNS.items():
    print(f"\n{field} — Pattern: {pattern}")
    for val in test_values[field]:
        match = "✅ VALID" if re.match(pattern, val) else "❌ INVALID"
        print(f"  {val:<25} → {match}")

### 6.2 The 10-Step Data Cleaning Checklist

In [ ]:
checklist = [
    "Profile the data — df.info(), df.describe(), df.isna().sum()",
    "Standardise nulls — map all missing markers to np.nan",
    "Fix data types — dates as datetime, IDs as string, amounts as float",
    "Validate and clean text fields — phone, email, name using regex",
    "Detect and resolve duplicates — exact then fuzzy",
    "Detect outliers — choose method based on distribution shape",
    "Decide action for each outlier — keep, cap, flag, or remove",
    "Standardise categorical values — case, synonyms, canonical mapping",
    "Scale numeric features — only before ML, never before EDA",
    "Build an audit log — record every transformation with counts",
]

print("📋 DATA CLEANING CHECKLIST")
print("=" * 60)
for i, item in enumerate(checklist, 1):
    print(f"  {i:2}. ☐ {item}")

---
## 🎯 Summary

| Chapter | Topic | Key Functions |
|---------|-------|---------------|
| 1 | **Regex** | `re.sub`, `re.match`, `re.search` |
| 2 | **Duplicates** | `df.duplicated()`, `fuzz.token_sort_ratio()`, `process.extractOne()` |
| 3 | **Outliers** | `stats.zscore()`, IQR, `IsolationForest`, `.clip()` |
| 4 | **Standardisation** | `MinMaxScaler`, `StandardScaler`, `RobustScaler`, `df.replace()` |
| 5 | **Pipeline** | `DataCleaner` class with audit log |

**Next steps:**
- Try `IterativeImputer` or `KNNImputer` from sklearn for advanced missing-value imputation
- Explore `recordlinkage` library for enterprise-scale fuzzy matching
- Use `great_expectations` for automated data quality checks in production pipelines